# Caldis — Getting Started

**Caldis** is a Python platform for modelling energy systems (heat pumps, gas
turbines, …) as assemblies of components.

The core idea, in one sentence:

> A component does **not** compute outputs. It declares the **equations** its
> physics must satisfy (called *residuals*), and a single solver finds the state
> that satisfies every component at once.

You never decide "who computes what" — you connect components, say what is
imposed (pressures, temperatures, flows), and the solver does the rest. The same
component works in a heat pump, a gas turbine, or a test bench, unchanged.

This notebook walks through:
1. building a small system (a compressor),
2. solving it,
3. reading the results,
4. exploring a component with `structure()` and `describe()`.

In [1]:
# --- imports ---
import numpy as np
from CoolProp.CoolProp import PropsSI

import caldis
from caldis.core.system import System
from caldis.components import FluidBoundary
from caldis.components import Compressor
from caldis.fluids import CoolPropBackend
from caldis.solvers import solve          # one-call workflow (DOF -> start -> solve)

print("Caldis version:", getattr(caldis, "__version__", "unknown"))

Caldis version: 0.2.0


## 1. Building a system

We size a **compressor** taking air at 1 bar / 15 °C and delivering it at 6 bar.

Three pieces:
- a **`FluidBoundary`** `src` that imposes the suction state (`p`, `T`) and the mass
  flow (`mdot`). Sign convention: `mdot > 0` enters a component, so a source that
  *sends* fluid out uses a negative `mdot`.
- the **`SizingCompressor`** itself (isentropic efficiency `eta_is`),
- a **`FluidBoundary`** `sink` that imposes only the discharge pressure.

We connect them with `connect(port_a, port_b)`. A `FluidBoundary` with a `backend`
knows its fluid; each component port carries its own backend too.

In [2]:
be = CoolPropBackend("Air")

p_suction, T_suction, m = 1.0e5, 288.15, 2.0    # 1 bar, 15 C, 2 kg/s
p_discharge = 6.0e5                             # 6 bar

comp = Compressor("comp", be, eta_is=0.85)
src  = FluidBoundary("src",  p=p_suction, T=T_suction, mdot=-m, backend=be)
sink = FluidBoundary("sink", p=p_discharge, backend=be)
sys_ = System("compressor_demo")
sys_.add(comp, src, sink)
sys_.connect(src.port, comp.inl)
sys_.connect(comp.out, sink.port)

# Optional START HINT: give the (hot) discharge port a rough temperature.
# Start hints are not the answer -- just a reasonable starting point for the solver.
# They must be set BEFORE init_starts (which solve() runs for us).
sys_.set_start_port(comp.out, p=p_discharge, T=500.0)

## 2. Solving

`solve()` runs the whole workflow and reports each stage:
1. **DOF** — is the system square (as many equations as unknowns)? A structural check.
2. **start point** — can every residual be evaluated at the starting guess?
3. **sanity** — physical warnings (non-blocking).
4. **solve** — the numerical solution (homotopy).

On success you get a one-line `✓ SOLVED`. On failure, `solve()` prints diagnostics
that name the offending equation or variable.

In [3]:
print(src.params["specs"])

{'p': 100000.0, 'T': 288.15, 'mdot': -2.0}


In [4]:
sol = solve(sys_)

✓ DOF: square (12 unknowns)
✓ start point: evaluable
⚠ 1 start-sanity warning(s):
   - comp: p_out (1.00 bar) must be > p_in (1.00 bar) for compression
  (non-blocking: the solver may still converge; shown in case it fails.)
✓ SOLVED (|F_scaled|=5.8e-10)


## 3. Reading the results

After solving, every port exposes its state on demand:
- `port.p`, `port.mdot`, `port.h` — the raw solved variables (SI: Pa, kg/s, J/kg),
- `port.T` (kelvin), `port.T_C` (°C), `port.x` (vapour quality), `port.s` (entropy),
  all recomputed from `(p, h)` when you ask.

Components also expose **derived outputs** via `eval_output(name)` (here the shaft
power `W`). And `system.port_table()` tabulates every port at once — the quickest
way to inspect a whole result.

In [5]:
print(f"suction    : {comp.inl.T_C:6.1f} °C   {comp.inl.p/1e5:.2f} bar")
print(f"discharge  : {comp.out.T_C:6.1f} °C   {comp.out.p/1e5:.2f} bar")
print(f"mass flow  : {comp.inl.mdot:.3f} kg/s")
print(f"power Wdot : {comp.Wdot/1e3:.1f} kW  (negative = absorbed)")
print(f"pressure ratio : {comp.pressure_ratio:.2f}")

sys_.port_table()

suction    :   15.0 °C   1.00 bar
discharge  :  239.3 °C   6.00 bar
mass flow  : 2.000 kg/s
power Wdot : -455.2 kW  (negative = absorbed)
pressure ratio : 6.00


p_bar           T         T_C      h_kJkg   s_kJkgK    x  \
component port                                                             
comp      inl     1.0  288.150000   15.000000  414.377832  3.849952  2.0   
          out     6.0  512.442633  239.292633  641.998550  3.918827  2.0   
src       port    1.0  288.150000   15.000000  414.377832  3.849952  2.0   
sink      port    6.0  512.442633  239.292633  641.998550  3.918827  2.0   

                     rho  mdot  
component port                  
comp      inl   1.209506   2.0  
          out   4.070694  -2.0  
src       port  1.209506  -2.0  
sink      port  4.070694   2.0

## 4. Exploring a component

Before wiring a component you don't know, two tools tell you what it offers:

- **`structure()`** — the *names* that exist: internal variables, ports, and any
  sub-components. Use it to know what to connect and what to read.
- **`describe()`** — the *values* after a solve: each port's state and every derived
  output.

`structure(depth=...)` limits how deep it expands into sub-components (a plain
component like the compressor is a leaf — nothing to expand).

In [6]:
# what does a compressor look like?
comp.structure()

print()
# what are its values after the solve?
comp.describe();

Compressor 'comp'
  port   inl  (Air)   -> .p .mdot .h
  port   out  (Air)   -> .p .mdot .h
  outputs ['Wdot', 'pressure_ratio']

=== Compressor('comp') ===
  ports (p [bar], mdot [kg/s], T [°C]):
    inl        p=1.000  mdot=+2  h=414.4kJ/kg  T=15.00
    out        p=6.000  mdot=-2  h=642.0kJ/kg  T=239.29
  outputs:
    Wdot           = -455241
    pressure_ratio = 6


## Next steps

You now know the full loop: **build → connect → impose → solve → read**.

Larger systems follow the same pattern. A few things you will meet:
- **Heat exchangers** (`SizingHX`, channels `hot`/`cold`) need distinct inlet/outlet
  temperatures as start hints, so the log-mean temperature difference is defined.
- **Closed loops** (e.g. a heat-pump cycle) need a `close_loop=True` connection and
  system-level constraints (`add_constraint`) for the pressure levels.
- Build **incrementally**: add one component at a time and `solve()` after each — the
  diagnostics then point at exactly what each new component needs.